### TradaBoostR2

This notebook runs TradaBoostR2 using the adapt package. For me Python 3.9 along with Tensorflow 2.15 worked!

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

from adapt.instance_based import TrAdaBoostR2, TwoStageTrAdaBoostR2
from sklearn.metrics import mean_squared_error, mean_absolute_error

import itertools

In [2]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Reshape
from tensorflow.keras.optimizers import Adam

def get_model():
    model = Sequential()
    model.add(Dense(12, activation='relu', input_shape=(30,)))
    model.add(Dense(12, activation='relu'))
    model.add(Dense(12, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer=Adam(1e-3), loss='mean_squared_error')
    return model

In [3]:
test_size = 0.93 #0.8or 0.93
seed_list = [1,2,3,4,5]

predictor_columns = ['pzabovezmean', 'pzabove2', 'zq5', 'zq10',
    'zq15', 'zq20', 'zq25', 'zq30', 'zq35', 'zq40', 'zq45', 'zq50', 'zq55',
    'zq60', 'zq65', 'zq70', 'zq75', 'zq80', 'zq85', 'zq90', 'zq95',
    'zpcum1', 'zpcum2', 'zpcum3', 'zpcum4', 'zpcum5', 'zpcum6', 'zpcum7',
    'zpcum8', 'zpcum9'
    ]

target_column = 'Dgv'

data_latvia = pd.read_csv(r'datasets/rs_lettland.csv', index_col=[0])
train_size = int((1-test_size)*len(data_latvia))
train_size

132

In [4]:
from tensorflow.keras.callbacks import Callback

class SavePrediction(Callback):
    """
    Callbacks which stores predicted
    labels in history at each epoch.
    """
    def __init__(self):
        self.X = np.linspace(-0.7, 0.6, 100).reshape(-1, 1)
        self.custom_history_ = []
        super().__init__()

    def on_epoch_end(self, batch, logs={}):
        """Applied at the end of each epoch"""
        predictions = self.model.predict_on_batch(self.X).ravel()
        self.custom_history_.append(predictions)

In [5]:
#ablation study for transfertreeboost Gaussian errors, with gaussian source domain errors
ablation_transfer_tradaboost_real = pd.DataFrame(columns = ['seed', 'method',
                                   'n_estimators', 'lr', 'epochs', 'rmse', 'mae'])

n_estimators_list = [5,15,25]
lr_list = [0.05, 0.1, 0.15]
epochs_list = [10, 20, 30]

# --- Step 2: Create full parameter grid ---
param_grid = list(itertools.product(
    n_estimators_list,
    lr_list,
    epochs_list
))

# --- Step 3: Sample random combinations ---
#sampled_configs = random.sample(param_grid, n_samples)

for seed in seed_list:

    #data from Svedala
    data_sweden = pd.read_csv(r'datasets/rs_sweden.csv', index_col=[0])


    



    

    #evaluate and rain on latvia instead (keep naming for simplicity)
    data_latvia = pd.read_csv(r'datasets/rs_lettland.csv', index_col=[0])
    data_latvia = data_latvia.rename(columns = {'H_AVERAGE': 'Hgv', 'D_AVERAGE': 'Dgv', 'VOLUME': 'Volume'})
    data_train, data_temp = train_test_split(data_latvia, test_size=test_size, random_state=seed)
    data_val, data_test = train_test_split(data_temp, test_size=0.5, random_state=seed)

    #"General" base dataset (to use for transfer)
    X_source_train = np.array(data_sweden[predictor_columns])
    y_source_train = np.array(data_sweden[target_column])

    #Specific train and test set
    X_target_train = np.array(data_train[predictor_columns])
    y_target_train = np.array(data_train[target_column])

    X_target_val = np.array(data_val[predictor_columns])
    y_target_val = np.array(data_val[target_column])

    X_target_test = np.array(data_test[predictor_columns])
    y_target_test = np.array(data_test[target_column])

    print(len(X_target_train), len(X_target_val), len(X_target_test))
    for config in param_grid:
        n_estimators, lr, epochs = config


        #Test for all methods!!!!

        method = f'TradaBoostR2'
        model = TrAdaBoostR2(get_model(),
                     n_estimators=n_estimators, lr=lr)

        model.fit(X_source_train, y_source_train, X_target_train, y_target_train, epochs = epochs, batch_size=16, verbose=0)
        preds = model.predict(X_target_test)
        rmse = np.sqrt(mean_squared_error(preds, y_target_test))
        mae = mean_absolute_error(preds, y_target_test)

        ablation_transfer_tradaboost_real.loc[len(ablation_transfer_tradaboost_real)] = [seed, method, n_estimators, lr, epochs, rmse, mae]
        ablation_transfer_tradaboost_real.to_csv(f'ablation_transfer_tradaboost_real_{train_size}.csv')


C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_11824\3423787743.py:22: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data_sweden = pd.read_csv(r'datasets/rs_sweden.csv', index_col=[0])


133 883 884


Iteration 0 - Error: 0.1338
Iteration 1 - Error: 0.1337
Iteration 2 - Error: 0.1417
Iteration 3 - Error: 0.1494
Iteration 4 - Error: 0.1633
Iteration 0 - Error: 0.1181
Iteration 1 - Error: 0.1305
Iteration 2 - Error: 0.1114
Iteration 3 - Error: 0.1187
Iteration 4 - Error: 0.1606
Iteration 0 - Error: 0.1195
Iteration 1 - Error: 0.1575
Iteration 2 - Error: 0.1366
Iteration 3 - Error: 0.1463
Iteration 4 - Error: 0.1507
Iteration 0 - Error: 0.1255
Iteration 1 - Error: 0.1419
Iteration 2 - Error: 0.1364
Iteration 3 - Error: 0.1356
Iteration 4 - Error: 0.1766
Iteration 0 - Error: 0.1303
Iteration 1 - Error: 0.1789
Iteration 2 - Error: 0.1307
Iteration 3 - Error: 0.1296
Iteration 4 - Error: 0.1495
Iteration 0 - Error: 0.1376
Iteration 1 - Error: 0.1233
Iteration 2 - Error: 0.1512
Iteration 3 - Error: 0.1473
Iteration 4 - Error: 0.1269
Iteration 0 - Error: 0.1341
Iteration 1 - Error: 0.1635
Iteration 2 - Error: 0.1391
Iteration 3 - Error: 0.1516
Iteration 4 - Error: 0.1405
Iterat

C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_11824\3423787743.py:22: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data_sweden = pd.read_csv(r'datasets/rs_sweden.csv', index_col=[0])


Iteration 0 - Error: 0.1544
Iteration 1 - Error: 0.1385
Iteration 2 - Error: 0.1471
Iteration 3 - Error: 0.1188
Iteration 4 - Error: 0.1356
Iteration 0 - Error: 0.1295
Iteration 1 - Error: 0.1400
Iteration 2 - Error: 0.1177
Iteration 3 - Error: 0.1172
Iteration 4 - Error: 0.1412
Iteration 0 - Error: 0.1320
Iteration 1 - Error: 0.1176
Iteration 2 - Error: 0.1252
Iteration 3 - Error: 0.1536
Iteration 4 - Error: 0.1400
Iteration 0 - Error: 0.1319
Iteration 1 - Error: 0.1277
Iteration 2 - Error: 0.1301
Iteration 3 - Error: 0.1471
Iteration 4 - Error: 0.1617
Iteration 0 - Error: 0.1589
Iteration 1 - Error: 0.1198
Iteration 2 - Error: 0.1303
Iteration 3 - Error: 0.1270
Iteration 4 - Error: 0.1709
Iteration 0 - Error: 0.1066
Iteration 1 - Error: 0.1301
Iteration 2 - Error: 0.1242
Iteration 3 - Error: 0.1300
Iteration 4 - Error: 0.1375
Iteration 0 - Error: 0.1525
Iteration 1 - Error: 0.1462
Iteration 2 - Error: 0.1584
Iteration 3 - Error: 0.1593
Iteration 4 - Error: 0.1442
Iteration 0 - Error:

KeyboardInterrupt: 

In [ ]:


model = TrAdaBoostR2(get_model(),
                     n_estimators=5, lr=0.1)

model.fit(Xs, ys, Xt_lab, yt_lab, epochs = 20, batch_size=16, verbose=0)
preds = model.predict(X_target_test)


np.sqrt(mean_squared_error(preds, y_target_test))

Iteration 0 - Cross-validation score: 5.9367 (0.2428)
Iteration 1 - Cross-validation score: 5.9367 (0.2428)
Iteration 2 - Cross-validation score: 5.1681 (0.3560)
Iteration 3 - Cross-validation score: 5.1097 (0.4708)
Iteration 4 - Cross-validation score: 5.1442 (0.3032)


8.068745422159658

In [ ]:
mae

5.253673427962643